### FINAL MODEL

In [2]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 2. Load the preprocessed dataset
data = pd.read_csv('/Users/arpitalonakadi/Downloads/employee_preprocessed_data.csv')

# 3. Create binary target using median
median_val = data['Absenteeism Time in Hours'].median()
data['Excessive Absenteeism'] = np.where(data['Absenteeism Time in Hours'] > median_val, 1, 0)

# 4. Drop original target column
data = data.drop(['Absenteeism Time in Hours'], axis=1)

# 5. Split into inputs and target
X = data.drop(['Excessive Absenteeism'], axis=1)
y = data['Excessive Absenteeism']

# 6. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 7. Train-test split
x_train, x_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=20)

# 8. Train the logistic regression model
model = LogisticRegression()
model.fit(x_train, y_train)

# 9. Predict probabilities
y_proba = model.predict_proba(x_test)[:, 1]

# 10. Apply custom threshold of 0.42
threshold = 0.42
y_pred_custom = np.where(y_proba >= threshold, 1, 0)

# 11. Evaluate the final model
print("=== Final Model with Threshold = 0.42 ===")
print(classification_report(y_test, y_pred_custom))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

=== Final Model with Threshold = 0.42 ===
              precision    recall  f1-score   support

           0       0.77      0.69      0.73        74
           1       0.69      0.77      0.73        66

    accuracy                           0.73       140
   macro avg       0.73      0.73      0.73       140
weighted avg       0.73      0.73      0.73       140

ROC AUC: 0.8001638001638002


### Thus this model is our final model 

- **Higher recall for Class 1 (absentees):**  
  - Final model recall = **0.77** vs Baseline recall = **0.70**  
  - This means the final model is better at identifying employees who are likely to be absent.

- **Improved ROC AUC score:**  
  - Final model ROC AUC = **0.800** vs Baseline ROC AUC = **0.792**  
  - Indicates better discriminatory power of the model.

- **Better balance between precision and recall:**  
  - Precision and recall are more evenly distributed (0.77 vs 0.69), giving a better F1-score balance.
  
- **Improved business utility:**  
  - In our use case, identifying most of the absentees is more critical than perfectly predicting both classes.
  - A higher recall for absentee class supports this objective.

- **Threshold tuning allows better control:**  
  - Custom threshold (0.42) lets us prioritize recall without sacrificing too much precision.

In [14]:
# 1. Convert scaled test data back to DataFrame (for readability)
X_test_df = pd.DataFrame(x_test, columns=X.columns)

# 2. Add predicted probability and prediction based on threshold
X_test_df['Predicted_Probability'] = y_proba
X_test_df['Final_Prediction'] = y_pred_custom

# 3. Add actual target (ground truth) for comparison
X_test_df['Actual_Label'] = y_test.values

# 4. View the predictions
X_test_df.head(10)  # You can change 10 to view more records

,Reason_1,Reason_2,Reason_3,Reason_4,Month Value,Transportation Expense,Daily Work Load Average,Body Mass Index,Children,Predicted_Probability,Final_Prediction,Actual_Label
0,-0.577350,-0.092981,-0.314485,0.821365,1.324766,-0.654143,-0.082083,1.002633,-0.919030,0.261465,0,0
1,-0.577350,-0.092981,-0.314485,0.821365,0.753746,1.036026,0.560476,-0.408580,-0.019280,0.411652,1,1
2,-0.577350,-0.092981,-0.314485,0.821365,1.324766,0.190942,0.305783,2.649049,-0.019280,0.632472,1,1
3,-0.577350,-0.092981,-0.314485,0.821365,-0.959313,-0.654143,-1.240355,1.002633,-0.919030,0.187934,0,0
4,1.732051,-0.092981,-0.314485,-1.217485,0.182726,0.568211,-0.806331,-0.878984,2.679969,0.938678,1,1
5,1.732051,-0.092981,-0.314485,-1.217485,-0.959313,-0.654143,-1.240355,1.002633,-0.919030,0.692569,1,1
6,1.732051,-0.092981,-0.314485,-1.217485,0.182726,-0.503235,-0.806331,-0.408580,0.880469,0.704439,1,1
7,-0.577350,-0.092981,3.179797,-1.217485,-0.673803,0.387122,-0.637953,1.237836,0.880469,0.918934,1,1
8,-0.577350,-0.092981,-0.314485,0.821365,-1.530333,0.040034,0.919937,-0.643782,-0.019280,0.195441,0,0
9,-0.577350,-0.092981,-0.314485,0.821365,0.182726,-0.654143,-0.806331,1.002633,-0.919030,0.239677,0,1


In [13]:
import pickle

# Save the trained logistic regression model
with open('/Users/arpitalonakadi/Downloads/final_logistic_model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)

# Save the scaler to the same location
with open('/Users/arpitalonakadi/Downloads/final_scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

print("✅ Model and scaler exported successfully to Downloads!")

✅ Model and scaler exported successfully to Downloads!


###  Model Deployment Preparation

We have successfully saved both the **final logistic regression model** and the **custom scaler**.  
This allows for easy reuse in the future — anyone can now load the model and scaler, feed in their preprocessed dataset, and obtain predictions effortlessly.

This setup ensures the model is production-ready and can be integrated into dashboards, automated systems, or other business tools for real-time absenteeism prediction.